## NEW FIX


In [0]:
# ============================================================
# Movie Attendance Clustering — category 1 / 2 / 3
# K-Means (MLlib) | PySpark / Databricks
# Features: total_attendance, run_weeks, avg_ticket_revenue
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

# ── 0. Config ────────────────────────────────────────────────
TRANSACTION_TABLE = "cpx_dataanalytics_gold.customer360.c360_f_transaction"
FILM_TABLE        = "cpx_dataanalytics_gold.customer360.cpx_d_film"
GENRE = "cpx_dataanalytics_gold.customer360.film_composite_genres"
K_RANGE           = range(3, 9)   # test k = 2 through 8
SEED              = 42

# ── 1. Load tables ───────────────────────────────────────────
txn  = spark.table(TRANSACTION_TABLE)
film = spark.table(FILM_TABLE)
genre = spark.table(GENRE)

# ── 2. Attendance from transactions ──────────────────────────
attendance = (
    txn
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("VisitDateId") > '20221201')
    .groupBy("film_id")
    .agg(
        F.sum("Quantity").alias("total_attendance"),
        F.avg("Net_Revenue").alias("avg_ticket_revenue"),
    )
    .filter(F.col("avg_ticket_revenue")<100)
    .filter(F.col("total_attendance")>0)
)

# ── 3. Flighting data from film table ────────────────────────
flighting = (
    film
    .join(genre, on="film_id", how="inner")
    .select("film_id", "title_id", "title_name", genre.genre_hierarchy_1, "theatrical_release_date")
)

# ── 4. Join & build feature set ──────────────────────────────
features_df = (
    flighting
    .join(attendance, on="film_id", how="inner")
    .select(
        "title_id", "title_name", genre.genre_hierarchy_1, "theatrical_release_date","total_attendance", "avg_ticket_revenue"
    )
    .filter(F.col("total_attendance").isNotNull())
    .filter(F.col("avg_ticket_revenue").isNotNull())
)

features_df.cache()  # cache — reused for every k trial
features_df.count()  # force cache

In [0]:
features_df.filter(F.col("avg_ticket_revenue")>100).display()

In [0]:
# ── 5. Assemble & scale features (reused across all k trials) ─
assembler = VectorAssembler(
    inputCols=["total_attendance", "avg_ticket_revenue"],
    outputCol="features_raw"
)
scaler = MinMaxScaler(inputCol="features_raw", outputCol="features")

prep_pipeline = Pipeline(stages=[assembler, scaler])
prep_model    = prep_pipeline.fit(features_df)
scaled_df     = prep_model.transform(features_df).cache()   # cache — reused for every k

# ── 6. Find optimal k (Elbow + Silhouette) ───────────────────
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette"
)

k_results = []

print("── K Search ─────────────────────────────────")
print(f"{'k':>4} │ {'WSSSE':>14} │ {'Silhouette':>12}")
print("─────┼────────────────┼─────────────────")

for k in K_RANGE:
    km = KMeans(
        featuresCol="features",
        predictionCol="cluster",
        k=k,
        seed=SEED,
        initMode="k-means||",
        maxIter=100,
        tol=1e-6,
    )
    km_model   = km.fit(scaled_df)
    predicted  = km_model.transform(scaled_df)

    wssse      = km_model.summary.trainingCost      # within-cluster sum of squared errors
    silhouette = evaluator.evaluate(predicted)

    k_results.append({"k": k, "wssse": wssse, "silhouette": silhouette})
    print(f"{k:>4} │ {wssse:>14,.1f} │ {silhouette:>12.4f}")

# ── 7. Pick best k ───────────────────────────────────────────
# Primary:  highest silhouette score (measures cluster separation)
# Tiebreak: elbow — largest drop in WSSSE relative to next k
best_by_silhouette = max(k_results, key=lambda x: x["silhouette"])

# Elbow: largest relative WSSSE drop between consecutive k values
wssse_values = [r["wssse"] for r in k_results]
elbow_drops  = [
    (wssse_values[i] - wssse_values[i + 1]) / wssse_values[i]
    for i in range(len(wssse_values) - 1)
]
elbow_k = k_results[elbow_drops.index(max(elbow_drops))]["k"]

print(f"\nBest k by silhouette : k = {best_by_silhouette['k']}  "
      f"(score = {best_by_silhouette['silhouette']:.4f})")
print(f"Best k by elbow      : k = {elbow_k}")

# Use silhouette as the primary signal; flag if they disagree
OPTIMAL_K = best_by_silhouette["k"]
if OPTIMAL_K != elbow_k:
    print(f"\n⚠  Methods disagree — defaulting to silhouette (k={OPTIMAL_K}). "
          f"Review the table above and override OPTIMAL_K if needed.")
else:
    print(f"\n✓  Both methods agree on k = {OPTIMAL_K}")

# ── 8. Final clustering with optimal k ───────────────────────
print(f"\n── Running final model with k = {OPTIMAL_K} ────────────")

final_km = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=OPTIMAL_K,
    seed=SEED,
    initMode="k-means||",
    maxIter=100,
    tol=1e-6,
)
final_model = final_km.fit(scaled_df)
clustered   = final_model.transform(scaled_df)

final_silhouette = evaluator.evaluate(clustered)
print(f"Final silhouette score: {final_silhouette:.4f}")

# ── 9. Map cluster IDs → categorys (highest attendance = category 1) ─
cluster_rank = (
    clustered
    .groupBy("cluster")
    .agg(F.avg("total_attendance").alias("avg_attendance"))
    .withColumn(
        "category",
        F.row_number().over(Window.orderBy(F.col("avg_attendance").desc()))
    )
    .select("cluster", "category")
)

In [0]:
# ── K Selection Visualization — Cineplex Brand (Light) ────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

# ── Cineplex brand palette — light theme ─────────────────────
CPX_BLUE       = "#003DA5"   # Cineplex primary blue
CPX_BLUE_LIGHT = "#2E6FD4"   # lines, markers
CPX_BLUE_FILL  = "#E8EEF9"   # bar fill (inactive)
CPX_GOLD       = "#C8A84B"   # Cineplex gold
CPX_GOLD_LIGHT = "#A07C20"   # gold text (darker so it reads on white)
CPX_GOLD_FILL  = "#FDF6E3"   # gold highlight fill

BG_PAGE        = "#FFFFFF"   # page / figure background
BG_PANEL       = "#F7F9FC"   # panel background
GRID_COLOR     = "#DDE3ED"   # gridlines
BORDER_COLOR   = "#C8D0DF"   # spine / divider

TEXT_PRIMARY   = "#0A1628"   # headings
TEXT_SECONDARY = "#4A5C7A"   # axis labels, subtitles
TEXT_DIM       = "#8A97AB"   # footer, minor labels

# ── Prep data ─────────────────────────────────────────────────
results_pd = pd.DataFrame(k_results)
k_vals     = results_pd["k"].tolist()
wssse_vals = results_pd["wssse"].tolist()
sil_vals   = results_pd["silhouette"].tolist()

wssse_norm = [(v - min(wssse_vals)) / (max(wssse_vals) - min(wssse_vals)) for v in wssse_vals]
sil_norm   = [(v - min(sil_vals))   / (max(sil_vals)   - min(sil_vals))   for v in sil_vals]

elbow_idx  = elbow_drops.index(max(elbow_drops))
sil_idx    = k_vals.index(best_by_silhouette["k"])

# ── Figure layout ─────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10), facecolor=BG_PAGE)
gs = gridspec.GridSpec(
    2, 2,
    figure=fig,
    hspace=0.62,   # slightly more vertical breathing room between rows
    wspace=0.38,
    top=0.83,      # unchanged — header still fits
    bottom=0.09,
    left=0.07,
    right=0.96,
)

# ── Shared axis styling ───────────────────────────────────────
def style_ax(ax, title, subtitle=""):
    ax.set_facecolor(BG_PANEL)
    for spine in ax.spines.values():
        spine.set_edgecolor(BORDER_COLOR)
        spine.set_linewidth(0.8)
    ax.grid(axis="y", color=GRID_COLOR, linestyle="-", linewidth=0.6, alpha=1)
    ax.grid(axis="x", visible=False)
    ax.tick_params(colors=TEXT_SECONDARY, labelsize=8.5, length=3, width=0.6)
    ax.set_xticks(k_vals)
    ax.xaxis.label.set_color(TEXT_SECONDARY)
    ax.yaxis.label.set_color(TEXT_SECONDARY)

    # Title — sits clearly above the panel
    ax.text(
        0, 1.15, title,
        transform=ax.transAxes,
        color=TEXT_PRIMARY, fontsize=10.5, fontweight="bold",
        va="bottom", ha="left",
    )
    # Subtitle — sits in the gap between title and panel top edge
    if subtitle:
        ax.text(
            0, 1.04, subtitle,
            transform=ax.transAxes,
            color=TEXT_DIM, fontsize=7.5,
            va="bottom", ha="left",
        )

# ─────────────────────────────────────────────────────────────
# Panel A — Elbow curve
# ─────────────────────────────────────────────────────────────
ax_elbow = fig.add_subplot(gs[0, 0])
style_ax(ax_elbow, "Elbow Curve", "WSSSE decreases as k increases — look for the bend")

# Area fill under curve
ax_elbow.fill_between(k_vals, wssse_vals, alpha=0.08, color=CPX_BLUE)

ax_elbow.plot(
    k_vals, wssse_vals,
    color=CPX_BLUE, linewidth=2.2,
    marker="o", markersize=7,
    markerfacecolor=BG_PAGE, markeredgecolor=CPX_BLUE, markeredgewidth=2,
    zorder=3,
)

ax_elbow.axvspan(k_vals[elbow_idx] - 0.4, k_vals[elbow_idx] + 0.4,
                 alpha=0.12, color=CPX_GOLD, zorder=1)
ax_elbow.axvline(k_vals[elbow_idx], color=CPX_GOLD, linestyle="--",
                 linewidth=1.3, alpha=0.9, zorder=2)

ax_elbow.annotate(
    f"  Elbow  k={elbow_k}",
    xy=(k_vals[elbow_idx], wssse_vals[elbow_idx]),
    xytext=(k_vals[elbow_idx] + 0.55, wssse_vals[elbow_idx] * 1.04),
    color=CPX_GOLD_LIGHT, fontsize=8, fontweight="bold",
    arrowprops=dict(arrowstyle="-|>", color=CPX_GOLD, lw=1.1),
)

ax_elbow.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1_000:.0f}K" if x < 1_000_000 else f"{x/1_000_000:.1f}M")
)
ax_elbow.set_xlabel("Number of Clusters  (k)", fontsize=8.5, labelpad=6)
ax_elbow.set_ylabel("WSSSE", fontsize=8.5, labelpad=6)

# ─────────────────────────────────────────────────────────────
# Panel B — Silhouette bar chart
# ─────────────────────────────────────────────────────────────
ax_sil = fig.add_subplot(gs[0, 1])
style_ax(ax_sil, "Silhouette Score", "Higher = better-separated clusters  (range: −1 to 1)")

bar_colors  = [CPX_GOLD      if i == sil_idx else CPX_BLUE_FILL  for i in range(len(k_vals))]
edge_colors = [CPX_GOLD_LIGHT if i == sil_idx else BORDER_COLOR  for i in range(len(k_vals))]

bars = ax_sil.bar(
    k_vals, sil_vals,
    color=bar_colors, edgecolor=edge_colors,
    linewidth=0.9, width=0.55, zorder=3,
)

for i, (bar, val) in enumerate(zip(bars, sil_vals)):
    ax_sil.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.004,
        f"{val:.3f}",
        ha="center", va="bottom", fontsize=7.8,
        color=CPX_GOLD_LIGHT if i == sil_idx else TEXT_SECONDARY,
        fontweight="bold" if i == sil_idx else "normal",
    )

ax_sil.axhline(max(sil_vals), color=CPX_GOLD, linestyle="--",
               linewidth=0.9, alpha=0.6)
ax_sil.set_ylim(0, max(sil_vals) * 1.18)
ax_sil.set_xlabel("Number of Clusters  (k)", fontsize=8.5, labelpad=6)
ax_sil.set_ylabel("Silhouette Score", fontsize=8.5, labelpad=6)

# ─────────────────────────────────────────────────────────────
# Panel C — Relative WSSSE drop
# ─────────────────────────────────────────────────────────────
ax_drop = fig.add_subplot(gs[1, 0])
style_ax(ax_drop, "Marginal WSSSE Improvement",
         "Largest bar = biggest gain from splitting one more cluster")

drop_k      = k_vals[:-1]
drop_colors = [CPX_BLUE      if i == elbow_idx else CPX_BLUE_FILL for i in range(len(drop_k))]
drop_edges  = [CPX_BLUE_LIGHT if i == elbow_idx else BORDER_COLOR  for i in range(len(drop_k))]

ax_drop.bar(drop_k, elbow_drops, color=drop_colors, edgecolor=drop_edges,
            linewidth=0.9, width=0.55, zorder=3)

for i, (x, drop) in enumerate(zip(drop_k, elbow_drops)):
    ax_drop.text(
        x, drop + max(elbow_drops) * 0.015,
        f"{drop:.3f}",
        ha="center", va="bottom", fontsize=7.8,
        color=CPX_BLUE if i == elbow_idx else TEXT_SECONDARY,
        fontweight="bold" if i == elbow_idx else "normal",
    )

ax_drop.set_xlabel("k  →  k+1", fontsize=8.5, labelpad=6)
ax_drop.set_ylabel("Relative Drop", fontsize=8.5, labelpad=6)
ax_drop.set_ylim(0, max(elbow_drops) * 1.22)

# ─────────────────────────────────────────────────────────────
# Panel D — Normalised overlay
# ─────────────────────────────────────────────────────────────
ax_ov = fig.add_subplot(gs[1, 1])
style_ax(ax_ov, "Signal Overlay (Normalised)",
         "Optimal k sits where silhouette peaks and WSSSE flattens")

ax_ov.fill_between(k_vals, wssse_norm, alpha=0.07, color=CPX_BLUE)
ax_ov.fill_between(k_vals, sil_norm,   alpha=0.07, color=CPX_GOLD)

ax_ov.plot(k_vals, wssse_norm,
           color=CPX_BLUE, linewidth=2, marker="o", markersize=6,
           markerfacecolor=BG_PAGE, markeredgecolor=CPX_BLUE, markeredgewidth=1.8,
           label="WSSSE  (↓ lower is better)", zorder=3)

ax_ov.plot(k_vals, sil_norm,
           color=CPX_GOLD, linewidth=2, marker="s", markersize=6,
           markerfacecolor=BG_PAGE, markeredgecolor=CPX_GOLD, markeredgewidth=1.8,
           label="Silhouette  (↑ higher is better)", zorder=3)

ax_ov.axvspan(OPTIMAL_K - 0.4, OPTIMAL_K + 0.4,
              alpha=0.08, color=CPX_BLUE, zorder=1)
ax_ov.axvline(OPTIMAL_K, color=CPX_BLUE, linestyle="--",
              linewidth=1.1, alpha=0.6, zorder=2)

ax_ov.annotate(
    f"  k = {OPTIMAL_K}  ✓",
    xy=(OPTIMAL_K, 1.0),
    xytext=(OPTIMAL_K + 0.3, 0.87),
    color=CPX_BLUE, fontsize=8.5, fontweight="bold",
    arrowprops=dict(arrowstyle="-|>", color=CPX_BLUE, lw=0.9),
)

ax_ov.legend(facecolor=BG_PAGE, edgecolor=BORDER_COLOR,
             labelcolor=TEXT_SECONDARY, fontsize=7.8,
             loc="lower right", framealpha=1)
ax_ov.set_xlabel("Number of Clusters  (k)", fontsize=8.5, labelpad=6)
ax_ov.set_ylabel("Normalised Score", fontsize=8.5, labelpad=6)
ax_ov.set_ylim(-0.05, 1.15)

# ── Header ────────────────────────────────────────────────────
agree   = OPTIMAL_K == elbow_k
verdict = (f"Both methods agree on  k = {OPTIMAL_K}" if agree else
           f"Silhouette → k={best_by_silhouette['k']}   |   Elbow → k={elbow_k}"
           f"   |   Defaulting to silhouette  (k={OPTIMAL_K})")
v_color = CPX_GOLD_LIGHT if agree else "#C0392B"
v_icon  = "✓" if agree else "⚠"

fig.text(0.07, 0.955, "K-Selection Analysis",
         color=TEXT_PRIMARY, fontsize=16, fontweight="bold", va="top")
fig.text(0.07, 0.927,
         "Movie Attendance Clustering  ·  Cineplex Entertainment  ·  "
         "Features: Total Attendance, Avg Ticket Revenue",
         color=TEXT_SECONDARY, fontsize=9, va="top")
fig.text(0.96, 0.948, f"{v_icon}  {verdict}",
         color=v_color, fontsize=8.5, fontweight="bold", va="top", ha="right")

# Blue rule under header
fig.add_artist(plt.Line2D(
    [0.07, 0.96], [0.905, 0.905],
    transform=fig.transFigure,
    color=CPX_BLUE, linewidth=2, alpha=0.9,
))

# Footer
fig.text(0.96, 0.025,
         f"Final Silhouette  {final_silhouette:.4f}   ·   Optimal k = {OPTIMAL_K}   ·   Cineplex Entertainment",
         color=TEXT_DIM, fontsize=7.5, ha="right", va="bottom")

# ── Save ──────────────────────────────────────────────────────
import os
save_dir = "/dbfs/FileStore/plots/cpx_media_strategy"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(f"{save_dir}/k_selection_cineplex.png", dpi=150,
            bbox_inches="tight", facecolor=BG_PAGE, edgecolor="none")
plt.show()

In [0]:
result = (
    clustered
    .join(cluster_rank, on="cluster", how="left")
    .select(
        "title_id", "title_name", "genre_hierarchy_1", "theatrical_release_date",
        "total_attendance",
        F.round("avg_ticket_revenue", 2).alias("avg_ticket_revenue"),
        "category",
    )
    .orderBy("category", F.col("total_attendance").desc())
)

# ── 10. Summary by category ──────────────────────────────────────
summary = (
    result
    .groupBy("category")
    .agg(
        F.count("title_id").alias("film_count"),
        F.round(F.avg("total_attendance"), 0).alias("avg_total_attendance"),
        F.round(F.avg("avg_ticket_revenue"), 2).alias("avg_ticket_rev"),
        F.min("total_attendance").alias("min_attendance"),
        F.max("total_attendance").alias("max_attendance"),
    )
    .orderBy("category")
)

print("\n── category Summary ─────────────────────────────")
summary.show(truncate=False)

print("\n── Film-Level Results ───────────────────────")
result.show(50, truncate=False)

scaled_df.unpersist()

# ── 11. Optionally write out ──────────────────────────────────
# result.write.mode("overwrite").saveAsTable("schema.film_attendance_categorys")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import pandas as pd
import numpy as np
import os

# ── Cineplex brand palette ────────────────────────────────────
CPX_BLUE       = "#003DA5"
CPX_BLUE_LIGHT = "#2E6FD4"
CPX_GOLD       = "#C8A84B"
CPX_GOLD_LIGHT = "#A07C20"

BG_PAGE        = "#FFFFFF"
BG_PANEL       = "#F7F9FC"
GRID_COLOR     = "#DDE3ED"
BORDER_COLOR   = "#C8D0DF"
TEXT_PRIMARY   = "#0A1628"
TEXT_SECONDARY = "#4A5C7A"
TEXT_DIM       = "#8A97AB"

# ── Dynamic category config based on OPTIMAL_K ────────────────────
# Anchored to Cineplex blue → gold; interpolates for any k
ANCHOR_COLORS = [
    "#003DA5",   # deep blue  (category 1 — highest attendance)
    "#2E6FD4",   # mid blue
    "#C8A84B",   # gold
    "#6AAF3D",   # green      (extra categorys if k > 3)
    "#C0392B",   # red
    "#8E44AD",   # purple
    "#16A085",   # teal
    "#E67E22",   # orange
]

category_LABELS_BASE = [
    "Blockbuster",
    "Mainstream",
    "Limited Run",
    "Niche",
    "Micro Release",
    "Specialty",
    "Archive",
    "Other",
]

def build_category_config(k):
    """
    Returns category_COLORS and category_LABELS dicts keyed 1..k.
    Uses anchored brand colors for k <= len(ANCHOR_COLORS),
    falls back to a blue→gold colormap interpolation beyond that.
    """
    colors, labels = {}, {}

    for category in range(1, k + 1):
        if category - 1 < len(ANCHOR_COLORS):
            colors[category] = ANCHOR_COLORS[category - 1]
        else:
            # Interpolate across the blue→gold spectrum for any overflow
            cmap = mcolors.LinearSegmentedColormap.from_list(
                "cpx", [CPX_BLUE, CPX_GOLD]
            )
            colors[category] = mcolors.to_hex(cmap((category - 1) / (k - 1)))

        label_base = category_LABELS_BASE[category - 1] if category - 1 < len(category_LABELS_BASE) else f"category {category}"
        labels[category] = f"category {category} — {label_base}"

    return colors, labels

category_COLORS, category_LABELS = build_category_config(OPTIMAL_K)

# ── Collect to pandas ─────────────────────────────────────────
plot_df = (
    result
    .select("title_name", "genre_hierarchy_1", "total_attendance", "avg_ticket_revenue", "category")
    .toPandas()
)

# ── Figure & axes ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8), facecolor=BG_PAGE)
ax.set_facecolor(BG_PANEL)

for spine in ax.spines.values():
    spine.set_edgecolor(BORDER_COLOR)
    spine.set_linewidth(0.8)

ax.grid(color=GRID_COLOR, linestyle="-", linewidth=0.6, alpha=1, zorder=0)
ax.tick_params(colors=TEXT_SECONDARY, labelsize=9, length=3, width=0.6)

# ── Plot each category ────────────────────────────────────────────
for category, group in plot_df.groupby("category"):
    ax.scatter(
        group["total_attendance"],
        group["avg_ticket_revenue"],
        c=category_COLORS[category],
        label=category_LABELS[category],
        alpha=0.82,
        s=70,
        edgecolors="white",
        linewidths=0.6,
        zorder=3,
    )

# ── Annotate top 5 per category ───────────────────────────────────
top_n = (
    plot_df
    .sort_values("total_attendance", ascending=False)
    .groupby("category")
    .head(5)
)

for _, row in top_n.iterrows():
    ax.annotate(
        row["title_name"],
        xy=(row["total_attendance"], row["avg_ticket_revenue"]),
        xytext=(7, 5),
        textcoords="offset points",
        fontsize=7,
        color=TEXT_SECONDARY,
        alpha=0.9,
        arrowprops=dict(arrowstyle="-", color=BORDER_COLOR, lw=0.6),
    )

# ── Axis formatting ───────────────────────────────────────────
ax.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1_000:.0f}K" if x < 1_000_000 else f"{x/1_000_000:.1f}M")
)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda y, _: f"${y:.0f}")
)
ax.set_xlabel("Total Attendance", color=TEXT_SECONDARY, fontsize=10, labelpad=8)
ax.set_ylabel("Avg Ticket Revenue ($)", color=TEXT_SECONDARY, fontsize=10, labelpad=8)

# ── Legend ────────────────────────────────────────────────────
legend = ax.legend(
    handles=[
        mpatches.Patch(color=category_COLORS[t], label=category_LABELS[t])
        for t in sorted(category_COLORS)
    ],
    facecolor=BG_PAGE,
    edgecolor=BORDER_COLOR,
    labelcolor=TEXT_SECONDARY,
    fontsize=9,
    loc="upper right",
    framealpha=1,
    title=f"Cluster categorys  (k = {OPTIMAL_K})",
    title_fontsize=8,
)
legend.get_title().set_color(TEXT_PRIMARY)
legend.get_title().set_fontweight("bold")

# ── Header ────────────────────────────────────────────────────
fig.text(0.07, 0.97, "Movie Attendance Clustering",
         color=TEXT_PRIMARY, fontsize=14, fontweight="bold", va="top")
fig.text(0.07, 0.945,
         f"Total Attendance vs Avg Ticket Revenue  ·  k = {OPTIMAL_K}  "
         f"·  Silhouette = {final_silhouette:.3f}  ·  Cineplex Entertainment",
         color=TEXT_SECONDARY, fontsize=8.5, va="top")

fig.add_artist(plt.Line2D(
    [0.07, 0.93], [0.928, 0.928],
    transform=fig.transFigure,
    color=CPX_BLUE, linewidth=2, alpha=0.9,
))

fig.text(0.93, 0.012,
         "Features: Total Attendance, Avg Ticket Revenue  ·  K-Means++  ·  Cineplex Entertainment",
         color=TEXT_DIM, fontsize=7.5, ha="right", va="bottom")

plt.subplots_adjust(top=0.90, bottom=0.09, left=0.08, right=0.93)

# ── Save ──────────────────────────────────────────────────────
save_dir = "/dbfs/FileStore/plots/cpx_media_strategy"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(f"{save_dir}/scatter_cineplex.png", dpi=150,
            bbox_inches="tight", facecolor=BG_PAGE, edgecolor="none")
plt.show()

In [0]:
# ── 12. Weekly MMM feature table ──────────────────────────────
SERIES_START = "2023-01-02"

# 12a. Title-week pairs: transactions grouped by title × week
film_to_title = (
    film
    .filter(F.col("performance_type") == "Film Presentation")
    .select("film_id", "title_id")
    .dropDuplicates()
)

weekly_title = (
    txn
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("VisitDateId") > "20221201")
    .withColumn("visit_date", F.to_date("VisitDateId", "yyyyMMdd"))
    .withColumn("week_date", F.date_trunc("week", F.col("visit_date")).cast("date"))
    .filter(F.col("week_date") >= F.lit(SERIES_START))
    .join(film_to_title, on="film_id", how="inner")
    .groupBy("title_id", "week_date")
    .agg(F.sum("Quantity").alias("weekly_attendance"))
    .filter(F.col("weekly_attendance") > 0)
    .join(result.select("title_id", "category").dropDuplicates(["title_id"]),
          on="title_id", how="inner")   # keep only clustered titles
    .cache()
)

# 12b. Week-level core metrics
category_share_cols = [f"cat{k}_share" for k in range(1, OPTIMAL_K + 1)]

week_core = (
    weekly_title
    .groupBy("week_date")
    .agg(
        F.countDistinct("title_id").alias("active_films"),
        *[
            F.round(
                F.sum(F.when(F.col("category") == k, F.lit(1)).otherwise(F.lit(0))) /
                F.countDistinct("title_id"),
                4
            ).alias(f"cat{k}_share")
            for k in range(1, OPTIMAL_K + 1)
        ],
        F.sum("weekly_attendance").alias("total_weekly_attendance"),
    )
)

# 12c. Genre hierarchy 3 per title — one row per title_id
#      Split composite underscore-delimited string into atomic genre tokens
#      e.g. "Action_Adventure_Animated" → ["Action", "Adventure", "Animated"]
title_h3 = (
    film_to_title
    .join(genre.select("film_id", "genre_hierarchy_3"), on="film_id", how="inner")
    .select("title_id", "genre_hierarchy_3")
    .dropDuplicates(["title_id"])
    .filter(F.col("genre_hierarchy_3").isNotNull())
    .withColumn("genre_tokens", F.split(F.col("genre_hierarchy_3"), "_"))
    .withColumn("genre_weight", F.lit(1.0) / F.size("genre_tokens"))
    .drop("genre_hierarchy_3")
)

# 12d. Genre intensity — weighted by movie count per week, not ticket sales
#      Each title contributes genre_weight / active_films to each of its genres
weekly_genre = (
    weekly_title.select("title_id", "week_date")
    .join(title_h3, on="title_id", how="inner")
    .join(week_core.select("week_date", "active_films"), on="week_date", how="left")
    .withColumn("genre", F.explode("genre_tokens"))
    .withColumn("genre", F.trim(F.col("genre")))
    .filter(F.col("genre") != "")
    .withColumn("genre_contribution", F.col("genre_weight") / F.col("active_films"))
    .dropDuplicates(["title_id", "week_date", "genre"])
)

# 12e. Pivot genres → one intensity column per atomic genre
def safe_name(g):
    return (g.lower()
             .replace(" ", "_")
             .replace("-", "_")
             .replace("&", "and")
             .replace("/", "_"))

genre_pivot = (
    weekly_genre
    .groupBy("week_date")
    .pivot("genre")
    .agg(F.round(F.sum("genre_contribution"), 4))
    .na.fill(0)
)

genre_raw_cols = [c for c in genre_pivot.columns if c != "week_date"]
for g in genre_raw_cols:
    genre_pivot = genre_pivot.withColumnRenamed(g, f"{safe_name(g)}_intensity")

genre_intensity_cols = sorted([f"{safe_name(g)}_intensity" for g in genre_raw_cols])
mmm_weekly = (
    week_core
    .join(genre_pivot, on="week_date", how="inner")
    .select(
        "week_date", "active_films",
        *category_share_cols,
        *genre_intensity_cols,
        "total_weekly_attendance",
    )
    .orderBy("week_date")
)

weekly_title.unpersist()


In [0]:
mmm_weekly.display()

In [0]:
genre_pivot.display()